In [2]:
import numpy as np


class AorticVessel:
    """
    ---------------------------------------------------------
    3D Aortic Vessel Geometry
    ---------------------------------------------------------

    Coordinate System

          y
          ^
          |
          |
          |
          +--------> x
         /
        /
       z  (Blood Flow Direction)

    Units
    -----
    Length : meter
    Radius : meter

    Geometry
    --------
    Healthy vessel with Gaussian stenosis.
    """

    def __init__(self,
                 length=0.06,                # 5 cm
                 healthy_radius=0.015,       # 12 mm
                 stenosis_center=0.03,      # 3 cm
                 stenosis_severity=0.70,     # 40 %
                 stenosis_width=0.006):      # 6 mm

        self.length = length

        self.R0 = healthy_radius

        self.zs = stenosis_center

        self.sigma = stenosis_width

        # Maximum radius reduction
        self.delta = self.R0 * stenosis_severity

    # =====================================================
    # Vessel Radius
    # =====================================================

    def radius(self, z):
        """
        Radius of vessel at longitudinal position z.
        """

        z = np.clip(z, 0.0, self.length)

        gaussian = np.exp(
            -((z - self.zs) ** 2) /
            (2 * self.sigma ** 2)
        )

        return self.R0 - self.delta * gaussian

    # =====================================================
    # Vessel Diameter
    # =====================================================

    def diameter(self, z):

        return 2.0 * self.radius(z)

    # =====================================================
    # Cross-sectional Area
    # =====================================================

    def area(self, z):

        r = self.radius(z)

        return np.pi * r ** 2

    # =====================================================
    # Distance from Robot to Vessel Center
    # =====================================================

    def radial_distance(self, position):

        x, y, _ = position

        return np.sqrt(x ** 2 + y ** 2)

    # =====================================================
    # Distance to Vessel Wall
    # =====================================================

    def wall_distance(self, position):
        """
        Positive  -> inside vessel

        Zero      -> touching wall

        Negative  -> outside vessel
        """

        _, _, z = position

        return self.radius(z) - self.radial_distance(position)

    # =====================================================
    # Collision Detection
    # =====================================================

    def check_collision(self, position):
        """
        True  -> collision

        False -> safe
        """

        x, y, z = position

        # Outside vessel length

        if z < 0:

            return True

        if z > self.length:

            return True

        radial = np.sqrt(x ** 2 + y ** 2)

        if radial >= self.radius(z):

            return True

        return False

    # =====================================================
    # Inside Vessel
    # =====================================================

    def is_inside(self, position):

        return not self.check_collision(position)

    # =====================================================
    # Vessel Volume (Approx.)
    # =====================================================

    def approximate_volume(self,
                           samples=500):

        z = np.linspace(
            0,
            self.length,
            samples
        )

        area = np.pi * self.radius(z) ** 2

        return np.trapz(area, z)

    # =====================================================
    # Print Geometry Information
    # =====================================================

    def info(self):

        print("=" * 50)

        print("Aortic Vessel Geometry")

        print("=" * 50)

        print(f"Length            : {self.length:.4f} m")

        print(f"Healthy Radius    : {self.R0:.4f} m")

        print(f"Stenosis Center   : {self.zs:.4f} m")

        print(f"Stenosis Width    : {self.sigma:.4f} m")

        print(f"Severity          : {100*self.delta/self.R0:.1f} %")

        print(f"Minimum Radius    : {self.radius(self.zs):.4f} m")

        print(f"Minimum Diameter  : {self.diameter(self.zs):.4f} m")

        print("=" * 50)

In [4]:
import numpy as np


class ValveModel:

    """
    Pulsatile Aortic Valve

    Output:
        Opening(t) ∈ [0,1]
    """

    def __init__(self,
                 heart_rate=130,
                 dt=0.01):

        self.heart_rate = heart_rate

        self.frequency = heart_rate / 60.0

        self.dt = dt

        self.time = 0.0

    # ------------------------------------

    def reset(self):

        self.time = 0.0

    # ------------------------------------

    def update(self):

        self.time += self.dt

    # ------------------------------------

    def opening(self):

        omega = 2*np.pi*self.frequency

        value = np.sin(omega*self.time)

        return np.maximum(value,0.0)

In [5]:
import numpy as np


class BloodFlow:
    """
    Blood Flow Model

    Components
    ----------
    1) Carreau viscosity
    2) Vessel geometry
    3) Pulsatile aortic valve
    4) Continuity equation
    """

    def __init__(self,
                 vessel,
                 valve,
                 eta0=0.046,
                 eta_inf=0.0035,
                 lam=3.313,
                 n=0.3568,
                 vmax=0.80):
        """
        Parameters
        ----------
        vessel : AorticVessel
        valve  : ValveModel

        vmax : float
            Maximum blood velocity in healthy vessel (m/s)
        """

        self.vessel = vessel
        self.valve = valve

        # Carreau parameters
        self.eta0 = eta0
        self.eta_inf = eta_inf
        self.lam = lam
        self.n = n
        self.delta_P_max = 600

        # Peak physiological velocity
        self.vmax = vmax
        self.length = 0.05

    # =====================================================
    # Carreau viscosity
    # =====================================================

    def viscosity(self, shear_rate):

        eta = (
            self.eta_inf +
            (self.eta0 - self.eta_inf) *
            (1 + (self.lam * shear_rate) ** 2)
            ** ((self.n - 1) / 2)
        )

        return eta

    # =====================================================
    # Local blood speed
    # =====================================================

    def speed(self,
              position,
              shear_rate):

        ####################################################
        # Vessel Geometry
        ####################################################

        z = position[2]

        radius = self.vessel.radius(z)

        healthy_radius = self.vessel.R0

        ####################################################
        # Valve opening
        ####################################################

        opening = self.valve.opening()

        ####################################################
        # Geometry amplification
        #
        # Continuity equation:
        #
        # V2 = V1 * (A1/A2)
        #
        # Since A = πR²
        #
        # V = Vmax * (R0/R)^2
        ####################################################

        geometry_factor = (healthy_radius / radius) ** 2
        geometry_factor = np.clip(geometry_factor,1.0,2.0)

        ####################################################
        # Carreau effect
        ####################################################

        eta = self.viscosity(shear_rate)

        viscosity_factor = self.eta_inf / eta

        ####################################################
        # Final speed
        ####################################################

        delta_P = self.delta_P_max * opening
        
        
        poiseuille_speed = (delta_P * radius**2) / (8 * eta * self.length)
        
        
        speed =  poiseuille_speed 
        
        '''
        speed = (
            self.vmax *
            opening *
            geometry_factor *
            viscosity_factor
        )
        '''
        return speed

    # =====================================================
    # Blood velocity vector
    # =====================================================

    def velocity(self,
                 position,
                 shear_rate):

        speed = self.speed(
            position,
            shear_rate
        )
        speed = speed * 0.05
        
        return np.array([
            0.0,
            0.0,
            -speed
        ])

    # =====================================================
    # Debug information
    # =====================================================

    def info(self,
             position,
             shear_rate):

        z = position[2]

        radius = self.vessel.radius(z)

        speed = self.speed(
            position,
            shear_rate
        )
'''
        print("=" * 60)
        print("Blood Flow")
        print("=" * 60)
        print(f"Position         : {position}")
        print(f"Radius           : {radius:.6f} m")
        print(f"Valve Opening    : {self.valve.opening():.3f}")
        print(f"Shear Rate       : {shear_rate:.2f}")
        print(f"Viscosity        : {self.viscosity(shear_rate):.6f} Pa.s")
        print(f"Blood Speed      : {speed:.4f} m/s")
        print("=" * 60)

        '''

'\n        print("=" * 60)\n        print("Blood Flow")\n        print("=" * 60)\n        print(f"Position         : {position}")\n        print(f"Radius           : {radius:.6f} m")\n        print(f"Valve Opening    : {self.valve.opening():.3f}")\n        print(f"Shear Rate       : {shear_rate:.2f}")\n        print(f"Viscosity        : {self.viscosity(shear_rate):.6f} Pa.s")\n        print(f"Blood Speed      : {speed:.4f} m/s")\n        print("=" * 60)\n\n        '

In [6]:
import numpy as np


class Microrobot:

    LEFT = 0
    RIGHT = 1
    UP = 2
    DOWN = 3
    FORWARD = 4
    BACKWARD = 5
    HOVER = 6

    def __init__(self,
                 speed=0.020,
                 dt=0.01):

        self.speed = speed
        self.dt = dt
        self.k = 0.5
        self.flow_resistance = 0.35

        self.reset()

    # -----------------------------

    def reset(self,
              start_position=None):
        self.velocity = np.zeros(3)

        if start_position is None:

            self.position = np.array([
                0.0,
                0.0,
                0.055
            ], dtype=float)

        else:

            self.position = np.array(
                start_position,
                dtype=float
            )

        return self.position.copy()

    # -----------------------------

    def step(self, action,blood_velocity):

        direction = np.zeros(3)

        if action == self.LEFT:
            direction = np.array([-1,0,0])

        elif action == self.RIGHT:
            direction = np.array([1,0,0])

        elif action == self.UP:
            direction = np.array([0,1,0])

        elif action == self.DOWN:
            direction = np.array([0,-1,0])

        elif action == self.FORWARD:
            direction = np.array([0,0,1])

        elif action == self.BACKWARD:
            direction = np.array([0,0,-1])

        elif action == self.HOVER:
            direction = np.zeros(3)

        blood_speed = np.linalg.norm(blood_velocity)

 

        
        #adaptive_speed = self.speed * (1.0 + self.k * blood_speed)
        #adaptive_speed = np.clip(adaptive_speed,self.speed,2.0 * self.speed)
        #flow_resistance = 1 / (1 + 5*np.sqrt(blood_speed))
       # flow_resistance = np.clip(flow_resistance, 0.25, 1.0)
        #blood_motion = blood_velocity * self.dt * flow_resistance
        


        

        
        #control_motion = direction * adaptive_speed * self.dt
        control_motion = direction * self.speed * self.dt

        blood_motion = blood_velocity * self.dt*0.25

        self.position += control_motion + 0.4*blood_motion  

        #print("Robot speed:", self.speed)
        #print("Blood speed:", np.linalg.norm(blood_velocity))
        #print("Blood motion:", np.linalg.norm(blood_motion))
        #print("Control motion:", np.linalg.norm(control_motion))

#        print(self.position)
      

        return self.position.copy()



        

In [7]:
'''
        control_force = direction * self.speed
        blood_force = blood_velocity
        k_resist = 0.5

        resist_force = -k_resist * blood_force
       # resist_force = -(0.5 + 0.9*np.tanh(blood_speed))

        net_force = (control_force+ blood_force + resist_force)
        acceleration = net_force
        self.velocity += acceleration * self.dt
        self.position += self.velocity * self.dt

        

        vmax = 0.015

        speed = np.linalg.norm(self.velocity)

        if speed > vmax:

            self.velocity = (self.velocity/ speed
            * vmax)


'''

'\n        control_force = direction * self.speed\n        blood_force = blood_velocity\n        k_resist = 0.5\n\n        resist_force = -k_resist * blood_force\n       # resist_force = -(0.5 + 0.9*np.tanh(blood_speed))\n\n        net_force = (control_force+ blood_force + resist_force)\n        acceleration = net_force\n        self.velocity += acceleration * self.dt\n        self.position += self.velocity * self.dt\n\n        \n\n        vmax = 0.015\n\n        speed = np.linalg.norm(self.velocity)\n\n        if speed > vmax:\n\n            self.velocity = (self.velocity/ speed\n            * vmax)\n\n\n'

## Aorta Environment, Env.py

In [8]:
#import gymnasium as gym
#from gymnasium import spaces

from gym import Env, spaces
import numpy as np
from collections import deque




class AorticValveEnv(Env):

    #metadata = {"render_modes": ["human"]}

    ########################################################

    def __init__(
            self,
            history_length=5,
            max_steps=1000):

        super().__init__()

        ####################################################
        # Components
        ####################################################

        self.vessel = AorticVessel()
        self.valve = ValveModel()

        self.blood = BloodFlow(
            vessel=self.vessel,
            valve=self.valve
        )

        self.robot = Microrobot()
        self.collision_count = 0
        self.total_collision_count = 0
        ####################################################
        # Goal (center of stenosis)
        ####################################################

        self.goal = np.array(
            [
                0.0,
                0.0,
                0.03
            ],
            dtype=np.float32
        )
        #print(self.goal)

        ####################################################

        self.history_length = history_length

        self.history = deque(
            maxlen=history_length
        )

        ####################################################

        self.max_steps = max_steps

        self.current_step = 0

        ####################################################

        self.shear_rate = 100.0

        ####################################################
        # Action Space
        ####################################################

        self.action_space = spaces.Discrete(7)

        ####################################################
        # Observation Space
        #
        # blood velocity (3)
        # goal vector (3)
        # distance (1)
        #
        ####################################################

        self.obs_dim = 7

        self.observation_space = spaces.Box(

            low=-np.inf,

            high=np.inf,

            shape=(history_length*self.obs_dim,),

            dtype=np.float32
        )

    ########################################################

    def _get_observation(self):

        position = self.robot.position

        radius = self.vessel.radius(position[2])

        blood_velocity = self.blood.velocity(

            position,

            self.shear_rate
        )

        goal_vector = self.goal-position

        distance = np.linalg.norm(goal_vector)

        observation = np.concatenate([

            blood_velocity,

            goal_vector,

            np.array([distance])

        ])

        return observation.astype(np.float32)

    ########################################################

    def reset(self):

        self.current_step = 0

        #self.robot.reset()
        self.valve.reset()
        self.collision_count = 0

        self.history.clear()

        
        start_position = np.array([

         np.random.uniform(-0.002,0.002),
         np.random.uniform(-0.002,0.002),
        0.055

    ])

        self.robot.reset(start_position)

        obs = self._get_observation()

        for _ in range(self.history_length):

            self.history.append(obs)

        state = np.concatenate(

            list(self.history)

        )

        

        return state

        ########################################################
    # Reward Function
    ########################################################

    def _compute_reward(
            self,
            distance,
            collision,
            goal_reached):

        reward = 1 / (distance + 1e-6)

        # Collision Penalty
        if collision:
            reward -= 15.0

        # Goal Reward
        if goal_reached:
            reward += 20.0

        # Small step penalty
        reward -= 0.01
        #reward = np.clip(reward, -100.0, 100.0)
        return reward

    ########################################################
    # Episode Termination
    ########################################################

    def _is_done(
            self,
            collision,
            goal_reached):

        if collision:
            return True

        if goal_reached:
            return True

        return False

    ########################################################
    # Step Function
    ########################################################

    def step(self, action):

        ####################################################
        # Increase step counter
        ####################################################

        self.current_step += 1
        self.valve.update()
        ####################################################
        # Current Position
        ####################################################

        current_position = self.robot.position.copy()

        ####################################################
        # Blood Flow
        ####################################################

        radius = self.vessel.radius(current_position)

        opening = self.valve.opening()
        blood_velocity = self.blood.velocity(
            radius,
            self.shear_rate
        )
        
        ####################################################
        # Move Robot
        ####################################################

        self.robot.step(
            action,
            blood_velocity
        )

        ####################################################
        # New Position
        ####################################################

        new_position = self.robot.position.copy()

        ####################################################
        # Collision Detection
        ####################################################

        collision = self.vessel.check_collision(
            new_position
        )

        if collision:
            #self.collision_count += 1
            self.total_collision_count += 1
        ####################################################
        # Distance to Goal
        ####################################################

        goal_vector = self.goal - new_position

        distance = np.linalg.norm(goal_vector)

        ####################################################
        # Goal Check
        ####################################################

        goal_reached = distance < 0.003

        ####################################################
        # Reward
        ####################################################

        reward = self._compute_reward(
            distance,
            collision,
            goal_reached
        )

        ####################################################
        # Termination
        ####################################################

        terminated = self._is_done(
            collision,
            goal_reached
        )

        truncated = (
            self.current_step >= self.max_steps
        )
        done = terminated or truncated
        ####################################################
        # Observation
        ####################################################

        observation = self._get_observation()

        self.history.append(observation)

        state = np.concatenate(
            list(self.history)
        )

        ####################################################
        # Extra Information
        ####################################################

        info = {

            "position": new_position,

            "distance": distance,

            "collision": collision,

            "goal_reached": goal_reached,

            "blood_velocity": blood_velocity,
            "collision_count": self.total_collision_count,
            "valve_opening": self.valve.opening(),
            "wall_distance": self.vessel.wall_distance(new_position)

        }
        """"
        print("="*70)
        print("Step            :", self.current_step)
        print("Action          :", action)
        print("Robot Position  :", new_position)
        print("Robot Radius    :", np.linalg.norm(new_position[:2]))
        print("wall distance   :", self.vessel.wall_distance(new_position))
        print("collision       :", collision)
        print("blood velocity  :", blood_velocity)
        print("valve opening   :", self.valve.opening())
        print("="*70)
        """
  
  
        
        
        
        
        return (
            state,
            reward,
            done,
            info
        )
########################################################

    #def render(self):
    #    pass



In [9]:
'''
        print("="*70)
        
        print(f"Step               : {self.current_step}")
        
        print(f"Robot Position      : {new_position}")
        
        print(f"Robot Radius        : {np.linalg.norm(new_position[:2]):.6f}")
        
        print(f"Vessel Radius       : {self.vessel.radius(new_position[2]):.6f}")
        
        print(f"Distance To Wall    : {self.vessel.wall_distance(new_position):.6f}")
        
        print(f"Valve Opening       : {self.valve.opening():.3f}")
        
        print(f"Blood Velocity      : {blood_velocity}")
        
        print(f"Blood Speed         : {np.linalg.norm(blood_velocity):.6f}")
        
        print(f"Collision           : {collision}")
        
        print(f"Reward              : {reward:.4f}")
        
        print("="*70)
'''

'\n        print("="*70)\n        \n        print(f"Step               : {self.current_step}")\n        \n        print(f"Robot Position      : {new_position}")\n        \n        print(f"Robot Radius        : {np.linalg.norm(new_position[:2]):.6f}")\n        \n        print(f"Vessel Radius       : {self.vessel.radius(new_position[2]):.6f}")\n        \n        print(f"Distance To Wall    : {self.vessel.wall_distance(new_position):.6f}")\n        \n        print(f"Valve Opening       : {self.valve.opening():.3f}")\n        \n        print(f"Blood Velocity      : {blood_velocity}")\n        \n        print(f"Blood Speed         : {np.linalg.norm(blood_velocity):.6f}")\n        \n        print(f"Collision           : {collision}")\n        \n        print(f"Reward              : {reward:.4f}")\n        \n        print("="*70)\n'

In [10]:
import matplotlib.pyplot as plt
from stable_baselines3 import PPO,A2C

## Risk-Sensitive PPO Model

In [11]:

import os
import numpy as np
import torch as T
import torch.nn as nn
import torch.optim as optim
from torch.distributions.categorical import Categorical

class PPOMemory:
    def __init__(self, batch_size):
        self.states = []
        self.probs = []   #  log-prob
        self.vals = []
        self.actions = []
        self.rewards = []
        self.dones = []

        self.batch_size = batch_size

    def generate_batches(self):
        n_states = len(self.states)
        batch_start = np.arange(0, n_states, self.batch_size)
        indices = np.arange(n_states, dtype=np.int64)
        np.random.shuffle(indices)
        batches = [indices[i:i+self.batch_size] for i in batch_start]

        return np.array(self.states),\
               np.array(self.actions),\
               np.array(self.probs),\
               np.array(self.vals),\
               np.array(self.rewards),\
               np.array(self.dones),\
               batches

    def store_memory(self, state, action, log_prob, val, reward, done):
        self.states.append(state)
        self.actions.append(action)
        self.probs.append(log_prob)  # 
        self.vals.append(val)
        self.rewards.append(reward)
        self.dones.append(done)

    def clear_memory(self):
        self.states = []
        self.probs = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.vals = []

class ActorNetwork(nn.Module):
    def __init__(self, n_actions, input_dims, alpha,hidden_size=64,
                 fc1_dims=64, fc2_dims=64, chkpt_dir='./chkpt'):
        super(ActorNetwork, self).__init__()

        os.makedirs(chkpt_dir, exist_ok=True)
        self.checkpoint_file = os.path.join(chkpt_dir, 'actor_torch_ppo.pt')

        self.hidden_size= hidden_size

        self.lstm = nn.LSTM(input_size=input_dims[0],
                            hidden_size=hidden_size,
                            batch_first=True)

        
        self.actor = nn.Sequential(
            nn.ReLU(),
            nn.Linear(hidden_size, fc1_dims),
            nn.ReLU(),
            nn.Linear(fc1_dims, fc2_dims),
            nn.ReLU(),
            nn.Linear(fc2_dims, n_actions)
            
        )

        self.optimizer = optim.Adam(self.parameters(), lr=alpha)
        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
        self.to(self.device)

    def forward(self, state, hidden=None):

        if state.dim() == 2:
            # 
            state = state.unsqueeze(1)

        h0 = T.zeros(1, state.size(0), self.lstm.hidden_size).to(self.device)
        c0 = T.zeros(1, state.size(0), self.lstm.hidden_size).to(self.device)    

        lstm_out, hidden = self.lstm(state, (h0,c0))
        last_output = lstm_out[:, -1, :]  # 

     
        logits = self.actor(last_output)

        
        dist = Categorical(logits=logits)  

        
        return dist

    def save_checkpoint(self):
        T.save(self.state_dict(), self.checkpoint_file)

    def load_checkpoint(self):
        self.load_state_dict(T.load(self.checkpoint_file, map_location=self.device))

class CriticNetwork(nn.Module):
    def __init__(self, input_dims, alpha, hidden_size=64,fc1_dims=64, fc2_dims=64, chkpt_dir='./chkpt'):
        super(CriticNetwork, self).__init__()

        os.makedirs(chkpt_dir, exist_ok=True)
        self.checkpoint_file = os.path.join(chkpt_dir, 'critic_torch_ppo.pt')
        self.hidden_size = hidden_size

        self.lstm = nn.LSTM(input_size=input_dims[0],
                            hidden_size=hidden_size,
                            batch_first=True)
        
        self.critic = nn.Sequential(
            nn.ReLU(),
            nn.Linear(hidden_size, fc1_dims),
            nn.ReLU(),
            nn.Linear(fc1_dims, fc2_dims),
            nn.ReLU(),
            nn.Linear(fc2_dims, 1)
        )

        self.optimizer = optim.Adam(self.parameters(), lr=alpha)
        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
        self.to(self.device)

    def forward(self, state):
      
       if state.dim() == 2:
          state = state.unsqueeze(1)


       h0 = T.zeros(1, state.size(0), self.lstm.hidden_size).to(self.device)
       c0 = T.zeros(1, state.size(0), self.lstm.hidden_size).to(self.device)    

       lstm_out, hidden = self.lstm(state, (h0,c0))
       last_output = lstm_out[:, -1, :]
       
       value = self.critic(last_output)


       return value

    def save_checkpoint(self):
        T.save(self.state_dict(), self.checkpoint_file)

    def load_checkpoint(self):
        self.load_state_dict(T.load(self.checkpoint_file, map_location=self.device))

class Agent:
    def __init__(self, n_actions, input_dims, gamma=0.99, alpha=0.0003, gae_lambda=0.95,
                 policy_clip=0.2, batch_size=64, n_epochs=10, risk_beta=1.0, risk_alpha=0.05):
        self.gamma = gamma
        self.policy_clip = policy_clip
        self.n_epochs = n_epochs
        self.gae_lambda = gae_lambda
        self.risk_beta = risk_beta  # 
        self.risk_alpha = risk_alpha  # 

        self.actor = ActorNetwork(n_actions, input_dims, alpha)
        self.critic = CriticNetwork(input_dims, alpha)
        self.memory = PPOMemory(batch_size)

    def remember(self, state, action, log_prob, val, reward, done):
        # state باید numpy array یا مشابه باشد
        self.memory.store_memory(state, action, log_prob, val, reward, done)

    def save_models(self):
        print('... saving models ...')
        self.actor.save_checkpoint()
        self.critic.save_checkpoint()

    def load_models(self):
        print('... loading models ...')
        self.actor.load_checkpoint()
        self.critic.load_checkpoint()

    def choose_action(self, observation):
        observation = np.array(observation, dtype=np.float32)
        state = T.tensor([observation], dtype=T.float).to(self.actor.device)

        dist = self.actor(state)
        value = self.critic(state)

        action = dist.sample()
        log_prob = dist.log_prob(action)

        action_item = int(action.item())
        log_prob_item = float(log_prob.item())
        value_item = float(value.item())

        return action_item, log_prob_item, value_item

    def compute_risk_sensitive_reward(self, advantages):
        
        beta = self.risk_beta

        scaled = -beta * advantages

        scaled = np.clip(scaled, -20, 20)

        risk_advantage = advantages * np.exp(scaled)



        

        #exp_values = np.exp(-beta * returns)

        #risk_return = -(1.0/beta) * np.log(np.mean(exp_values))

        return risk_advantage       
     
        # 
        #return np.log1p(np.exp(self.risk_beta * reward))* (1 / self.risk_beta)

    def learn(self):
        if len(self.memory.states) == 0:
            return

        for _ in range(self.n_epochs):
            state_arr, action_arr, old_logprob_arr, vals_arr, reward_arr, dones_arr, batches = \
                self.memory.generate_batches()

            # \
            reward_arr = np.array(reward_arr, dtype=np.float32)
            returns = np.zeros_like(reward_arr, dtype=np.float32)

            G = 0.0

            for t in reversed(range(len(reward_arr))):

                if dones_arr[t]:
                    G = 0.0

                G = reward_arr[t] + self.gamma * G

                returns[t] = G

          


            #transformed_rewards = np.array([self.compute_risk_sensitive_reward(r) for r in returns], dtype=np.float32)

            # 
            values = np.array(vals_arr, dtype=np.float32)

            # GAE )
            advantages = np.zeros_like(returns, dtype=np.float32)
            lastgaelam = 0.0
            # append a 0 to values to index t+1 safely
            values_ext = np.append(values, 0.0)
            for t in reversed(range(len(returns))):
                nonterminal = 1.0 - float(dones_arr[t])
                delta = returns[t] + self.gamma * values_ext[t+1] * nonterminal - values_ext[t]
                lastgaelam = delta + self.gamma * self.gae_lambda * nonterminal * lastgaelam
                advantages[t] = lastgaelam

            advantages = T.tensor(advantages, dtype=T.float).to(self.actor.device)
            values_tensor = T.tensor(values, dtype=T.float).to(self.actor.device)
#            print("Reward:", reward_arr[:10])
#            print("Risk Reward:", transformed_rewards[:10])
#           print("Advantage:", advantages[:10])

            for batch in batches:
                states = T.tensor(state_arr[batch], dtype=T.float).to(self.actor.device)
                old_log_probs = T.tensor(old_logprob_arr[batch], dtype=T.float).to(self.actor.device)
                actions = T.tensor(action_arr[batch], dtype=T.long).to(self.actor.device)

                
                
                
                
                
                dist = self.actor(states)
                critic_value = self.critic(states).squeeze()

                new_log_probs = dist.log_prob(actions)

                # نسبت احتمال: exp(new_log - old_log)
                prob_ratio = (new_log_probs - old_log_probs).exp()

                adv_batch = advantages[batch]
                
    
                
                
                
                
                risk_advantage = self.compute_risk_sensitive_reward(adv_batch.cpu().numpy())
                risk_advantage = T.tensor(risk_advantage,dtype=T.float).to(self.actor.device)
                  

                #adv_batch = risk_advantages[batch]
                weighted_probs = risk_advantage * prob_ratio
                weighted_clipped_probs = risk_advantage * T.clamp(prob_ratio, 1 - self.policy_clip, 1 + self.policy_clip)
                
                
               #print(
                 #   risk_advantage.min().item(),
                #    risk_advantage.max().item()
                #)
                
                #print(T.isnan(risk_advantage).any())
                #print(T.isinf(risk_advantage).any())
                                
                                
                
                
                
                
                
                actor_loss = -T.min(weighted_probs, weighted_clipped_probs).mean()

                # target returns = advantages + values (values from buffer)
                returns = (adv_batch + values_tensor[batch]).to(self.actor.device)
                returns = (returns - returns.mean()) / (returns.std() + 1e-8)

                critic_loss = (returns - critic_value) ** 2
                critic_loss = critic_loss.mean()

                total_loss = actor_loss + 0.5 * critic_loss

                for name, p in self.actor.named_parameters():

                    if p.grad is not None:

                        print(name,"Grad max:",p.grad.abs().max().item() )
           
               
              #  print("Target Return:", returns[:5])
              #  print("Critic Value:", critic_value[:5])
              #  print(actor_loss.item())
              #  print(critic_loss.item())
               # print("adv:",T.isnan(adv_batch).any(), T.isinf(adv_batch).any())
               # print("risk:",T.isnan(risk_advantage).any(),T.isinf(risk_advantage).any())
               # print("ratio:",T.isnan(prob_ratio).any(),T.isinf(prob_ratio).any())
               # print("returns:",T.isnan(returns).any(),T.isinf(returns).any())
               # print("critic:",T.isnan(critic_value).any(),T.isinf(critic_value).any())
            


    

   

                
                self.actor.optimizer.zero_grad()
                self.critic.optimizer.zero_grad()
                total_loss.backward()
                #T.nn.utils.clip_grad_norm_(self.actor.parameters(), 0.5)
                #T.nn.utils.clip_grad_norm_(self.critic.parameters(), 0.5)
                self.actor.optimizer.step()
                self.critic.optimizer.step()

        self.memory.clear_memory()


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt




                
if __name__ == '__main__':

    env =  AorticValveEnv()
    #env = DummyVecEnv([lambda: SimpleBloodVesselEnvironment()])
    

    N = 20 
    batch_size = 64
    n_epochs = 4
    alpha = 0.0003            

    agent = Agent(n_actions=7, batch_size=batch_size,
                  alpha=alpha, n_epochs=n_epochs,
                  input_dims=env.observation_space.shape ,risk_beta= 0.8)

    #model = PPO("MlpPolicy",env,verbose=0)


   


    
import numpy as np
import time
import matplotlib.pyplot as plt


csv_file = "training_results.csv"  
n_games = 300  # تعداد اپیزودها
best_score = 0
score_history = []
episode_lengths = []
reward_stddev = []
time_to_goal_list = []
successful_episodes = 0
learn_iters = 0
n_steps = 0
collision_history = []
trajectories = {}



# حلقه آموزش
for i in range(n_games):
    observation = env.reset()  # 
    done = False
    score = 0
    t0 = time.time()  #
    steps = 0
    rewards_in_episode = []  # 
    trajectory = [] 
    #model.learn(total_timesteps=1000)
    while not done:
        action, prob, val = agent.choose_action(observation)
        #print(type(observation))
        
        #action, _states = model.predict(observation)
        observation_, reward, done, info = env.step(action)

        #normalized_reward = reward_normalizer.normalize(reward)

        #clipped_reward = reward_clipping(normalized_reward)

        #agent.remember(observation, action, prob, val, reward, done)
        n_steps += 1
        steps += 1
        #reward = np.clip(reward, -10.0, 10.0)
        agent.remember(observation, action, prob, val, reward,done)


        score += np.tanh(reward / 100)
        
        #score += reward
        
        

        rewards_in_episode.append(reward)  # 
        
        trajectory.append(
        info["position"].copy()
    )
        # 
        #if len(agent.memory.states) >= batch_size:
            #agent.learn()

        observation = observation_

    # ذخیره نتایج هر اپیزود
    t1 = time.time()
    score_history.append(score)
    episode_lengths.append(steps)
    reward_stddev.append(np.std(score_history[-10:]))  # 
    collision_history.append(info["collision_count"])
   
    
    if info["goal_reached"]:   
        
        successful_episodes += 1
        time_to_goal_list.append(t1 - t0)

    avg_score = np.mean(score_history[-100:])

    # ذخیره مقادیر در فایل CSV بعد از هر اپیزود
    if 'trajectories' not in locals():
        trajectories = {}
    trajectories[i] = trajectory
    # نمایش نتایج هر 10 اپیزود
    if (i + 1) % 1 == 0:
        print(f'\n[Episode {i + 1}/{n_games}] | '
              f'Score: {score:.2f} | '
              f'Avg Score: {avg_score:.2f} | '
              f'Std: {reward_stddev[-1]:.2f} | '
              f'Len: {steps} | '
              f'Successes: {successful_episodes} | '
              f'Time: {t1 - t0:.2f}s|'
              f"Collisions: {info['collision_count']:3d}")

episodes = list(range(1, n_games + 1))
plt.figure(figsize=(14, 6))

plt.subplot(1, 3, 1)
plt.plot(episodes, score_history, label='Reward per Episode')
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('Reward Trend')
plt.grid()

plt.subplot(1, 3, 2)
plt.plot(episodes, reward_stddev, label='Reward Std Dev', color='orange')
plt.xlabel('Episode')
plt.ylabel('Std Dev')
plt.title('Reward Standard Deviation')
plt.grid()

plt.subplot(1, 3, 3)
plt.plot(episodes, episode_lengths, label='Episode Length', color='green')
plt.xlabel('Episode')
plt.ylabel('Steps')
plt.title('Steps per Episode')
plt.grid()

plt.tight_layout()
plt.show()

# به‌روزرسانی نتایج نهایی
if time_to_goal_list:
    avg_time_to_goal = np.mean(time_to_goal_list)
    print(f"\n✅ Average time to reach goal (for successful episodes): {avg_time_to_goal:.2f} seconds")
print(f"🎯 Total Successful Episodes: {successful_episodes}/{n_games}")